<a href="https://colab.research.google.com/github/1021114Carlos/MIT_MM_Finance/blob/Finance-shop/Courses/Foundation%20of%20modern%20finance%20I/M4_fixed_income.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sympy as sp
import math
from IPython.display import display, Math
import numpy as np
import pandas as pd

(Q1)

In [ ]:
display(Math(r"B_p = \frac{1}{(1 + r_t)^t}"))

In [ ]:
def bootstrap_spot_rates(bonds: pd.DataFrame):

  bonds = bonds.copy().sort_values("T").reset_index(drop=True)
  bonds["coupon"] = bonds["coupon_rate"]*bonds["fv"]

  N = int(bonds["T"].max())
  DF = np.full(N + 1, np.nan)
  spot = np.full(N + 1, np.nan)

  # Boostrap
  for _, row in bonds.iterrows():
    T = int(row["T"])
    price = float(row["price"])
    C = float(row["coupon"])
    FV = float(row["fv"])

    if T == 1:
      # 1- year: price = (FV + C)*DF1
      DF[T] = price/(FV + C)
    else:
      # price = sum_{t=1}^{T-1} C*DF[t] + (FV+C)*DF[T]
      pv_coupons = C*np.nansum(DF[1:T])
      DF[T] = (price - pv_coupons)/(FV + C)

    # Convert DF_T -> spot r_T
    spot[T] = DF[T]**(-1/T) - 1

  bonds["DF"] = bonds["T"].map(lambda t: DF[int(t)])
  bonds["spot"] = bonds["T"].map(lambda t: spot[int(t)])
  return bonds

In [ ]:
bonds = pd.DataFrame({"T": [1, 2, 3], "price": [97.5, 96, 98],
                      "coupon_rate": [0, 0.03, 0.035],
                      "fv": [100, 100, 100],})

out = bootstrap_spot_rates(bonds)
print(out[["T", "price", "coupon_rate", "DF", "spot"]])


(Q2 a)

In [ ]:
def solve_arbitrage(face_value, prices, coupon_rates, maturities):
  PA, PB, PC, PD = prices
  cA, cB, cC, cD = coupon_rates
  TA, TB, TC, TD = maturities

  if not (TA == 1 and cA == 0.0):
    raise ValueError("This solver expects bond A to be a 1-year zero (t=1, coupon=0)")
  if not (cC == 0.0):
    raise ValueError("This solver expects bond C to be a zeor-coupon bond (coupon=0)")
  if not (TB == 2):
    raise ValueError("This solver expects bond B to have maturity t=2 (so it pins down df2)")
  if not (TC == 3 and TD == 3):
    raise ValueError ("This solver expects bonds C and D to have maturity t=3")

  CB = cB*face_value
  CD = cD*face_value

  DF1 = PA/face_value
  DF3 = cD*face_value

  DF2 = (PB - CB*DF1)/(face_value + CB)
  PD_star = CD*DF1 + CD*DF2 + (face_value + CD)*DF3
  mispricing = PD - PD_star

  y = CD/(face_value + CB)
  x = y*CB/face_value

  A_qty = (CD/face_value) - x
  B_qty = y
  C_qty = (face_value + CD)/face_value

  replica_cost = A_qty*PA + B_qty*PB + C_qty*PC
  profit_today = PD - replica_cost

  k = 100/profit_today
  scaled = {"A": A_qty*k, "B": B_qty*k, "C": C_qty*k, "D": -1.0*k}

  return {"DF1": DF1, "DF2": DF2, "DF3": DF3,
          "PD_star": PD_star, "mispricing_PD_minus_fair": mispricing,
          "replicating_portfolio": {"A": A_qty, "B": B_qty,"C": C_qty},
          "profit_today_per_1_short_D": profit_today,
          "scaled_100_today_portfolio": scaled}


face_value = float(input("Face value (e.g., 100): ").strip())
prices = list(map(float, input("Prices A, B, C, D: ").split(",")))
coupon_rates = list(map(float, input("Coupon rates A, B, C, D (decimals, 1% = 0.01): ").split(",")))
maturities = list(map(int, input("Maturities t for A, B, C, D (e.g., 1, 3, 4, 4):  ").split(",")))

res = solve_arbitrage(face_value, prices, coupon_rates, maturities)

print("\nDiscount factors:")
print(res["DF1"], res["DF2"], res["DF3"])

print("\nFair price of D and mispricing (market - fair): ")
print(res["PD_star"], res["mispricing_PD_minus_fair"])

print("\nReplicating portfolio for 1 unit of D (long these, short 1 D):")
print(res["replicating_portfolio"])
print("Profit today per 1 short D:", res["profit_today_per_1_short_D"])

print("\nPortfolio that yields +100 and 0 in the future (qty: + long, - short): ")
print(res["scaled_100_today_portfolio"])


(Q3)

In [ ]:
t = 5
bond_cr = 4.5/100
face_value = 100
bond_ytm = 3.15/100

pv = (face_value*bond_cr)*((1 - (1 + bond_ytm)**(-t))/bond_ytm) + (100/(1 + bond_ytm)**t)
pv
# for m in range(1, t-1):    # m = maturity
#   pv_coupons = (face_value*bond_cr)/(1 + bond_ytm)**m
# pv_m = (face_value*bond_cr + face_value)/

(Q4) Compouting YTM

In [ ]:
display(Math(r"B_{ytm} = \sum_{t=1}^{T} = \frac{C}{(1 + y)^t} + \frac{C + P}{(1 + y)^t}"))


In [ ]:
 # Compute bond price from spot rates
 # Solve ytm using numerical method because problem is nonlinear

def price_from_spots(FV, coupon_rate, r1, r2, r3):
  C = FV*coupon_rate
  P = C/(1+r1) + C/(1+r2)**2 + (FV + C)/(1 + r3)**3
  return P

def ytm_bisection(FV, coupon_rate, P, low=0.0, high=0.10, tol=1e-12, max_iter=200):
  C = FV*coupon_rate

  def f(y):
    return C/(1+y) + C/(1+y)**2 + (FV+C)/(1+y)**3 - P

  while f(low)*f(high) > 0:
    high *= 2
    if high > 100:
      raise ValueError("Could not bracket the YTM root.")

  for _ in range(max_iter):
    mid = (low + high)/2
    val = f(mid)
    if abs(val) < tol:
      return mid
    if f(low)*val <= 0:
      high = mid
    else:
      low = mid
  return mid

FV = 100
coupon_rate = 5.25/100
r1, r2, r3 = (1.1/100), (1.15/100), (1.5/100)

P = price_from_spots(FV, coupon_rate, r1, r2, r3)
y = ytm_bisection(FV, coupon_rate, P)

print("Price from spots:", P)
print("YTM:", y, "=>", 100*y, "%")



(Q5) Computing Bond Duration